In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!tar -xzf "/content/drive/MyDrive/CodecRobust/processed_upload.tar.gz" -C /content/
print("Done.")
!ls /content/uncompressed/dev/ | head -3

^C
Done.
LA_D_1006568.pt
LA_D_1008730.pt
LA_D_1010295.pt


In [5]:
!git clone https://github.com/roh1thbharathi/applied-dl-project-2.git
%cd /content/applied-dl-project-2
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/applied-dl-project-2/src')
print("Done.")

Cloning into 'applied-dl-project-2'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 58 (delta 15), reused 50 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 37.32 KiB | 4.66 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/applied-dl-project-2
Done.


In [3]:
import os
from pathlib import Path
import data_utils

original_init = data_utils.ASVspoof2019Preprocessed.__init__

def patched_init(self, asv_root, processed_root, split="train", contrastive=False):
    original_init(self, asv_root, processed_root, split, contrastive)
    available = set(
        f.replace('.pt', '')
        for f in os.listdir(Path(processed_root) / 'uncompressed' / split)
    )
    before = len(self.records)
    self.records = self.records[self.records['filename'].isin(available)].reset_index(drop=True)
    print(f"  [Filter] {split}: {before} → {len(self.records)} (matched preprocessed files)")

data_utils.ASVspoof2019Preprocessed.__init__ = patched_init
print("Patch applied.")

Patch applied.


In [9]:
ASV_ROOT       = "/content/asvspoof"
PROCESSED_ROOT = "/content"
print("Paths set.")

Paths set.


In [8]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving ASVspoof2019.LA.cm.train.trn.txt to ASVspoof2019.LA.cm.train.trn.txt
Saving ASVspoof2019.LA.cm.eval.trl.txt to ASVspoof2019.LA.cm.eval.trl.txt
Saving ASVspoof2019.LA.cm.dev.trl.txt to ASVspoof2019.LA.cm.dev.trl.txt
Uploaded: ['ASVspoof2019.LA.cm.train.trn.txt', 'ASVspoof2019.LA.cm.eval.trl.txt', 'ASVspoof2019.LA.cm.dev.trl.txt']


In [10]:
import os, shutil

os.makedirs("/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)
os.makedirs("/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)

for fname in ['ASVspoof2019.LA.cm.train.trn.txt',
              'ASVspoof2019.LA.cm.dev.trl.txt',
              'ASVspoof2019.LA.cm.eval.trl.txt']:
    src   = f"/content/applied-dl-project-2/{fname}"
    local = f"/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    drive = f"/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    shutil.move(src, local)
    shutil.copy(local, drive)

print("Done.")
!ls /content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/

Done.
ASVspoof2019.LA.cm.dev.trl.txt	 ASVspoof2019.LA.cm.train.trn.txt
ASVspoof2019.LA.cm.eval.trl.txt


In [11]:
  import shutil

# Backup original
shutil.copy('/content/applied-dl-project-2/src/model.py',
            '/content/applied-dl-project-2/src/model_backup.py')

# Read current file
with open('/content/applied-dl-project-2/src/model.py', 'r') as f:
    content = f.read()

# Make the 3 targeted changes
content = content.replace(
    "    def __init__(self, embed_dim=256, sinc_ch=70, sample_rate=16000, n_attn_heads=4):\n        super().__init__()\n        self.sinc",
    "    def __init__(self, embed_dim=256, sinc_ch=70, sample_rate=16000, n_attn_heads=4, use_temporal_attn=True):\n        super().__init__()\n        self.use_temporal_attn = use_temporal_attn\n        self.sinc"
)

content = content.replace(
    "        r    = self.temporal_attn(r)      # (B, 32, 128) — codec-robust focus",
    "        if self.use_temporal_attn:\n            r = self.temporal_attn(r)"
)

content = content.replace(
    "    def __init__(self, embed_dim=256, n_codec_classes=10, sample_rate=16000, n_attn_heads=4):\n        super().__init__()\n        # TemporalAttention is now INSIDE AASISTEncoder (on 32 nodes before pool)\n        self.encoder = AASISTEncoder(embed_dim=embed_dim, sample_rate=sample_rate,\n                                     n_attn_heads=n_attn_heads)",
    "    def __init__(self, embed_dim=256, n_codec_classes=10, sample_rate=16000, n_attn_heads=4, use_temporal_attn=True):\n        super().__init__()\n        self.encoder = AASISTEncoder(embed_dim=embed_dim, sample_rate=sample_rate,\n                                     n_attn_heads=n_attn_heads, use_temporal_attn=use_temporal_attn)"
)

with open('/content/applied-dl-project-2/src/model.py', 'w') as f:
    f.write(content)

# Reload
import sys
for mod in list(sys.modules.keys()):
    if mod in ['model', 'data_utils', 'evaluate']:
        del sys.modules[mod]

from model import CodecRobustDetector
m = CodecRobustDetector(use_temporal_attn=False)
print("Success. Params:", sum(p.numel() for p in m.parameters()))

Success. Params: 550948


In [14]:
import sys, os, importlib
from pathlib import Path

for mod in list(sys.modules.keys()):
    if mod in ['model', 'data_utils', 'evaluate']:
        del sys.modules[mod]

import data_utils
importlib.reload(data_utils)

_original_init = data_utils.ASVspoof2019Preprocessed.__init__

def patched_init(self, asv_root, processed_root, split="train", contrastive=False):
    _original_init(self, asv_root, processed_root, split, contrastive)
    available = set(f.replace('.pt','') for f in os.listdir(Path(processed_root)/'uncompressed'/split))
    before = len(self.records)
    self.records = self.records[self.records['filename'].isin(available)].reset_index(drop=True)
    print(f"  [Filter] {split}: {before} → {len(self.records)} (matched preprocessed files)")

data_utils.ASVspoof2019Preprocessed.__init__ = patched_init
print("Ready.")

Ready.


In [7]:
import shutil, os

os.makedirs("/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)

for fname in ['ASVspoof2019.LA.cm.train.trn.txt',
              'ASVspoof2019.LA.cm.dev.trl.txt',
              'ASVspoof2019.LA.cm.eval.trl.txt']:
    shutil.copy(
        f"/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols/{fname}",
        f"/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    )

print("Done.")

Done.


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA               = 0.5    # GRL on
BETA                = 0.1    # Contrastive on
USE_CONTRASTIVE     = True
USE_TEMPORAL_ATTN   = False  # temporal off
EPOCHS              = 30
BATCH_SIZE          = 16
LR                  = 3e-4
MAX_SAMPLES         = 3000
EMBED_DIM           = 256
SAVE_EVERY          = 1
RESULTS_DIR         = "/content/drive/MyDrive/CodecRobust/results/grl_contrastive_no_temporal"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

os.makedirs(RESULTS_DIR, exist_ok=True)

loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
)

model      = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES,
                                  use_temporal_attn=USE_TEMPORAL_ATTN).to(device)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
con_crit   = ContrastiveLoss(temperature=0.07)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

out_dir     = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER: {best_eer*100:.2f}%")
else:
    print("Starting fresh.")

def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels) + ALPHA * codec_crit(out["codec_logits"], codec_idx)

        if USE_CONTRASTIVE and "waveform2" in batch:
            wf2   = batch["waveform2"].to(device)
            out2  = model(wf2)
            B     = wf.size(0)
            pairs = torch.stack([out["embedding"], out2["embedding"]], dim=1).view(2*B, -1)
            loss  = loss + BETA * con_crit(pairs)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
Free GPU: 15.52 GB
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
Starting fresh.
Epoch 01/30 | EER=8.23% AUC=0.9710 | time=252.9s
  ✓ Best EER: 8.23% — saved
  ✓ Checkpoint saved: epoch 1
Epoch 02/30 | EER=7.72% AUC=0.9771 | time=255.8s
  ✓ Best EER: 7.72% — saved
  ✓ Checkpoint saved: epoch 2
Epoch 03/30 | EER=4.01% AUC=0.9901 | time=254.8s
  ✓ Best EER: 4.01% — saved
  ✓ Checkpoint saved: epoch 3
Epoch 04/30 | EER=33.88% AUC=0.7426 | time=255.1s
  ✓ Checkpoint saved: epoch 4
Epoch 05/30 | EER=13.00% AUC=0.9241 | time=254.6s
  ✓ Checkpoint saved: epoch 5
Epoch 06

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA               = 0.0    # GRL off
BETA                = 0.0    # Contrastive off
USE_CONTRASTIVE     = False
USE_TEMPORAL_ATTN   = False  # temporal off
EPOCHS              = 30
BATCH_SIZE          = 32
LR                  = 3e-4
MAX_SAMPLES         = 3000
EMBED_DIM           = 256
SAVE_EVERY          = 1
RESULTS_DIR         = "/content/drive/MyDrive/CodecRobust/results/vanilla"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
)

model      = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES,
                                  use_temporal_attn=USE_TEMPORAL_ATTN).to(device)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

out_dir     = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER: {best_eer*100:.2f}%")
else:
    print("Starting fresh.")

def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
Free GPU: 4.44 GB
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
Starting fresh.
Epoch 01/30 | EER=9.11% AUC=0.9572 | time=142.3s
  ✓ Best EER: 9.11% — saved
  ✓ Checkpoint saved: epoch 1
Epoch 02/30 | EER=8.06% AUC=0.9762 | time=142.4s
  ✓ Best EER: 8.06% — saved
  ✓ Checkpoint saved: epoch 2
Epoch 03/30 | EER=9.69% AUC=0.9693 | time=142.4s
  ✓ Checkpoint saved: epoch 3
Epoch 04/30 | EER=5.15% AUC=0.9884 | time=142.7s
  ✓ Best EER: 5.15% — saved
  ✓ Checkpoint saved: epoch 4
Epoch 05/30 | EER=8.65% AUC=0.9523 | time=142.3s
  ✓ Checkpoint saved: epoch 5
